In [0]:
import pyspark.sql.functions as func

In [0]:
df = spark.read.table("f1_racing.bronze.brz_races")

display(df.limit(4))

In [0]:
df = df.drop('url','fp1_date','fp1_time','fp2_date','fp2_time','fp3_date','fp3_time','quali_date','quali_time','sprint_date','sprint_time','source_fie','ingested_at')

In [0]:
display(df.limit(4))

In [0]:
# df = df.withColumn('date_time', func.concat(func.col('date'),func.lit(' '), func.col('time')).cast('timestamp'))\
#     .drop('date','time')

# display(df.limit(4))

In [0]:
df.groupBy('time').agg(func.count("circuitId")).show()

In [0]:
df = df.withColumn(
    "time_in_seconds", 
    func.hour("time") * 3600 + func.minute("time") * 60 + func.second("time")
)

df.filter(func.col('time')!= r'\N').groupBy('name').avg('time_in_seconds').show(truncate=False)

In [0]:
from pyspark.sql.window import Window

In [0]:
win = Window.partitionBy('name')
df = df.withColumn('time_in_seconds',func.when(func.col("time")== r'\N',43200)
              .otherwise(func.col('time_in_seconds')))


In [0]:
df = df.withColumn('avg_time',func.avg(func.col('time_in_seconds')).over(win))

In [0]:
import pyspark.sql.types as t
@func.udf(returnType=t.StringType())
def time_out_put(sec : float) :
    hours = int(sec // 3600)
    minutes = int((sec % 3600) // 60)
    seconds = int(sec % 60)

    microseconds = int((sec - int(sec)) * 1000000)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


In [0]:
df = df.withColumn('time',func.when(func.col('time')== r'\N',time_out_put(func.col('avg_time')))
              .otherwise(func.col('time')))

In [0]:
df = df.withColumn('date_time', func.concat(func.col("date"),func.lit(' '),func.col("time")))\
    .drop('avg_time','time','date','time_in_seconds')

df.show(5)

In [0]:
df.write.format('delta')\
    .mode('overwrite')\
    .option('mergeSchema',True)\
    .saveAsTable('f1_racing.silver.slv_races')